# Deploy the Reference H2O Endpoint

Register the reference bundle and pinned environment, deploy at zero traffic, inspect initialization logs, invoke directly, verify golden parity, and optionally promote traffic.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/04_deploy_managed_online_endpoint.ipynb` and the Azure ML managed endpoint examples.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes, ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    Environment,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["H2O_ENVIRONMENT_VERSION"]
ENDPOINT_NAME = os.environ["H2O_ENDPOINT_NAME"]
DEPLOYMENT_NAME = os.environ["H2O_DEPLOYMENT_NAME"]
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
DEPLOY = os.getenv("DEPLOY_H2O_ENDPOINT", "false").lower() in {"1", "true", "yes"}
PROMOTE = os.getenv("PROMOTE_H2O_TRAFFIC", "false").lower() in {"1", "true", "yes"}
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))

model_definition = Model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(BUNDLE_DIR),
    description="Validated workshop H2O binary-model bundle",
    tags={"workshop": "azureml-h2o", "model_format": "h2o_binary", "h2o_version": manifest["h2o_version"]},
)
environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    version=ENVIRONMENT_VERSION,
    image="mcr.microsoft.com/azureml/minimal-py312-inference:latest",
    conda_file=str(WORKSHOP_ROOT / "environment/h2o/online-conda.yaml"),
    description="OpenJDK 17 and exact H2O runtime for online scoring",
    tags={"workshop": "azureml-h2o", "h2o_version": manifest["h2o_version"]},
)
identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )
endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    auth_mode="aad_token",
    identity=identity,
    public_network_access=os.getenv("AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled"),
    description="Workshop H2O binary-model endpoint",
    tags={"workshop": "azureml-h2o"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}:{MODEL_VERSION}",
    environment=f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}",
    code_configuration=CodeConfiguration(code=str(WORKSHOP_ROOT / "src/h2o/online"), scoring_script="score.py"),
    instance_type=os.environ["AZUREML_ONLINE_INSTANCE_TYPE"],
    instance_count=1,
    app_insights_enabled=True,
    environment_variables={"WORKER_COUNT": "1", "H2O_NTHREADS": os.environ["H2O_NTHREADS"], "H2O_MAX_MEM_SIZE": os.environ["H2O_MAX_MEM_SIZE"]},
)

if DEPLOY:
    ml_client.models.create_or_update(model_definition)
    ml_client.environments.create_or_update(environment_definition)
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
    deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")
    logs = ml_client.online_deployments.get_logs(DEPLOYMENT_NAME, ENDPOINT_NAME, 100, container_type="inference-server")
    print("\n".join(logs.splitlines()[-25:]))

    golden_input = pd.read_csv(BUNDLE_DIR / "golden_input.csv")
    golden_expected = pd.read_csv(BUNDLE_DIR / "golden_expected.csv")
    request = {"input_data": {"columns": manifest["features"], "data": golden_input[manifest["features"]].values.tolist()}}
    request_path = WORKSHOP_ROOT / "outputs/h2o_reference_request.json"
    request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
    response = json.loads(ml_client.online_endpoints.invoke(endpoint_name=ENDPOINT_NAME, deployment_name=DEPLOYMENT_NAME, request_file=str(request_path)))
    np.testing.assert_allclose(golden_expected["predict"], response["predictions"], rtol=1e-6, atol=1e-6)
    print(f"Cloud parity passed for {len(response['predictions'])} rows.")

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged. Set PROMOTE_H2O_TRAFFIC=true to promote.")
else:
    print(f"Prepared {ENDPOINT_NAME}/{DEPLOYMENT_NAME}; set DEPLOY_H2O_ENDPOINT=true to deploy.")

## Expected Result

The cloud deployment reaches `Succeeded`, initialization logs show the H2O model loaded, direct invocation matches golden predictions, and traffic changes only when enabled.

Next: `05_submit_reference_scoring_pipeline.ipynb`.